# 02 — Preprocessing : Transformations & Class Weights

Dans ce notebook on met en place :
1. **Les transformations d'images** (resize, augmentation, normalisation)
2. **Les class weights** pour gérer le déséquilibre NORMAL/PNEUMONIA
3. **Les DataLoaders** — les objets qui alimenteront le modèle en images pendant l'entraînement

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

DATA_PATH = Path('../data/raw/chest_xray')
print('PyTorch version :', torch.__version__)
print('Dataset trouvé :', DATA_PATH.exists())

## 2. Définir les transformations

On définit des transformations **différentes** pour le train et pour val/test.

**Pourquoi différentes ?**
- Sur le **train** : on veut que le modèle voie des images variées → augmentation aléatoire
- Sur **val/test** : on veut évaluer le modèle sur des images propres → pas d'aléatoire

**Pipeline train :**
```
Image originale
    → Resize(256) en bicubique   # On agrandit légèrement
    → RandomCrop(224)            # On coupe aléatoirement 224×224
    → RandomHorizontalFlip       # On retourne parfois l'image
    → RandomRotation(±10°)       # Légère rotation aléatoire
    → ToTensor                   # Pixels (0-255) → tenseur (0.0-1.0)
    → Normalize                  # Centrage avec les stats ImageNet
```

**Pipeline val/test :**
```
Image originale
    → Resize(256) en bicubique
    → CenterCrop(224)            # On coupe le centre (fixe, pas aléatoire)
    → ToTensor
    → Normalize
```

In [ ]:
# Valeurs de normalisation standard ImageNet
# ResNet a été pré-entraîné avec ces valeurs → on doit les réutiliser
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),       # 50% de chances d'être retournée
    transforms.RandomRotation(degrees=10),         # Rotation entre -10° et +10°
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # Variation légère de luminosité
    transforms.ToTensor(),                         # Convertit en tenseur PyTorch
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

val_test_transforms = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

print('Transformations définies ✓')

## 3. Charger les datasets

`ImageFolder` est une classe de torchvision qui charge automatiquement les images
depuis un dossier organisé ainsi :
```
train/
  NORMAL/      ← classe 0
  PNEUMONIA/   ← classe 1
```
Elle associe automatiquement chaque image à sa classe selon le nom du sous-dossier.

In [ ]:
train_dataset = datasets.ImageFolder(DATA_PATH / 'train', transform=train_transforms)
val_dataset   = datasets.ImageFolder(DATA_PATH / 'val',   transform=val_test_transforms)
test_dataset  = datasets.ImageFolder(DATA_PATH / 'test',  transform=val_test_transforms)

print(f'Train : {len(train_dataset)} images')
print(f'Val   : {len(val_dataset)} images')
print(f'Test  : {len(test_dataset)} images')
print(f'Classes : {train_dataset.classes}')  # ['NORMAL', 'PNEUMONIA']
print(f'Index   : {train_dataset.class_to_idx}')  # {'NORMAL': 0, 'PNEUMONIA': 1}

## 4. Calculer les class weights

Formule :
```
poids_classe = total_images / (nb_classes × nb_images_dans_cette_classe)
```
La classe rare aura un poids élevé → le modèle y fera plus attention.

In [ ]:
# Compter les images par classe dans le train
labels = [label for _, label in train_dataset.samples]
n_normal    = labels.count(0)
n_pneumonia = labels.count(1)
n_total     = len(labels)
n_classes   = 2

weight_normal    = n_total / (n_classes * n_normal)
weight_pneumonia = n_total / (n_classes * n_pneumonia)

class_weights = torch.tensor([weight_normal, weight_pneumonia], dtype=torch.float)

print(f'NORMAL    : {n_normal} images  → poids = {weight_normal:.3f}')
print(f'PNEUMONIA : {n_pneumonia} images  → poids = {weight_pneumonia:.3f}')
print(f'\n→ Une erreur sur NORMAL compte {weight_normal/weight_pneumonia:.1f}x plus que sur PNEUMONIA')
print(f'class_weights = {class_weights}')

## 5. Créer les DataLoaders

Un **DataLoader** est un objet qui :
- Découpe le dataset en **mini-batches** (petits paquets d'images)
- **Mélange** les images à chaque époque (shuffle)
- Charge les images **en parallèle** pour ne pas faire attendre le modèle

**Pourquoi des mini-batches ?**
On ne peut pas donner toutes les 5000 images d'un coup au modèle — ça ne tiendrait pas en mémoire. On lui donne 32 images à la fois (batch_size=32).

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,       # Mélange à chaque époque
    num_workers=2,      # Chargement en parallèle (2 processus)
    pin_memory=True     # Optimisation mémoire
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,      # Pas besoin de mélanger pour l'évaluation
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

print(f'Batch size : {BATCH_SIZE} images')
print(f'Batches train : {len(train_loader)}')
print(f'Batches val   : {len(val_loader)}')
print(f'Batches test  : {len(test_loader)}')

## 6. Vérifier un batch

On charge un batch et on vérifie que tout a bien la bonne forme.

In [ ]:
images, labels = next(iter(train_loader))

print(f'Forme du batch images : {images.shape}')
print(f'  → [32 images, 3 canaux RGB, 224 pixels hauteur, 224 pixels largeur]')
print(f'Forme des labels : {labels.shape}')
print(f'Valeurs min/max après normalisation : {images.min():.2f} / {images.max():.2f}')
print(f'Labels dans ce batch : {labels.tolist()[:10]}...')
print(f'  (0 = NORMAL, 1 = PNEUMONIA)')

## 7. Visualiser l'effet de l'augmentation

On prend **une seule image** et on lui applique les transformations plusieurs fois
pour voir les différentes versions que le modèle recevra pendant l'entraînement.

In [ ]:
def denormalize(tensor):
    """Annule la normalisation pour pouvoir afficher l'image."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return torch.clamp(tensor * std + mean, 0, 1)

# Charger une image PNEUMONIA brute
sample_path = list((DATA_PATH / 'train' / 'PNEUMONIA').glob('*.jpeg'))[0]
raw_img = Image.open(sample_path).convert('RGB')

fig, axes = plt.subplots(2, 5, figsize=(18, 7))

# Ligne du haut : image originale + 4 versions augmentées
axes[0, 0].imshow(raw_img)
axes[0, 0].set_title('Originale', fontsize=11)
axes[0, 0].axis('off')

for i in range(1, 5):
    augmented = train_transforms(raw_img)
    img_display = denormalize(augmented).permute(1, 2, 0).numpy()
    axes[0, i].imshow(img_display)
    axes[0, i].set_title(f'Augmentation {i}', fontsize=11)
    axes[0, i].axis('off')

# Ligne du bas : même image mais en niveaux de gris pour mieux voir les détails
axes[1, 0].imshow(raw_img.convert('L'), cmap='gray')
axes[1, 0].set_title('Originale (gris)', fontsize=11)
axes[1, 0].axis('off')

for i in range(1, 5):
    augmented = train_transforms(raw_img)
    img_display = denormalize(augmented).permute(1, 2, 0).numpy()
    gray = np.mean(img_display, axis=2)
    axes[1, i].imshow(gray, cmap='gray')
    axes[1, i].set_title(f'Augmentation {i} (gris)', fontsize=11)
    axes[1, i].axis('off')

plt.suptitle('Effet de la data augmentation sur une radiographie PNEUMONIA', fontsize=13)
plt.tight_layout()
plt.savefig('../results/figures/augmentation_examples.png', dpi=150)
plt.show()

## 8. Sauvegarder la configuration

On sauvegarde les class_weights pour les réutiliser dans le notebook d'entraînement.

In [ ]:
import json

config = {
    'n_normal': n_normal,
    'n_pneumonia': n_pneumonia,
    'weight_normal': round(weight_normal, 4),
    'weight_pneumonia': round(weight_pneumonia, 4),
    'batch_size': BATCH_SIZE,
    'image_size': 224,
    'imagenet_mean': IMAGENET_MEAN,
    'imagenet_std': IMAGENET_STD
}

with open('../results/preprocessing_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Configuration sauvegardée dans results/preprocessing_config.json')
print(json.dumps(config, indent=2))

## 9. Conclusions

Ce qu'on a mis en place :

| Élément | Choix | Raison |
|---------|-------|--------|
| Resize | 256px bicubique | Qualité maximale, sans déformation |
| Crop | 224×224 | Taille standard ResNet |
| Augmentation | Flip + Rotation + Luminosité | Variété pour mieux généraliser |
| Normalisation | Stats ImageNet | Cohérence avec le pré-entraînement |
| Class weights | NORMAL ≈ 1.94, PNEUMONIA ≈ 0.67 | Compenser le déséquilibre |
| Batch size | 32 | Bon compromis mémoire/stabilité |

---
**Prochain notebook :** `03_baseline_cnn.ipynb` — on construit et entraîne un premier CNN simple.